# GW170817 PE — Fixed sky — Low-resolution grid — BayesWave-cleaned data (v4)

Parameter estimation of GW170817 with **sky location fixed** to the known EM counterpart (NGC 4993):
$$\alpha = 3.44616\;\mathrm{rad}, \quad \delta = -0.408084\;\mathrm{rad}$$

- **Waveform**: `mlgw_bns_jax` — JAX-based BNS approximant (neural-network surrogate of `mlgw_bns`)
- **Sampler**: `nuts_dynesty` — NUTS nested sampling (JAX + BlackJax)
- **Data**: 1024 s of GWOSC strain for H1, L1, V1 — L1 glitch subtracted via **DCC BayesWave** (BayesWave) data
- **Segment duration**: 128 s (used for FFT and PSD estimation)
- **Likelihood evaluation**: on a **uniform grid of 3000 points** in $[23, 2000]$ Hz ($\Delta f \approx 0.66$ Hz)

The key advantage of `mlgw_bns_jax` over FFT-based waveforms is that it can be evaluated on **any** frequency grid. We exploit this by resampling the detector data (strain FFT and PSD) onto a coarse uniform grid of 3000 points, drastically reducing the number of likelihood evaluations per sample (from ~253k to 3k frequency bins).

**Compatibility**: This notebook runs on **Google Colab** and **LIGO JupyterHub**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saulo-albuquerque-phys/nuts_dynesty/blob/main/test_gw170817/pe_mlgw_bns_jax_low_grid_v4.ipynb)

### Proper prior choices (following LVK conventions)

**11 sampled parameters** (RA and Dec fixed):

| Index | Parameter | Prior | Range | Notes |
|:---:|---|---|---|---|
| 0 | $d_L$ (luminosity distance) | Volumetric ($p \propto d_L^2$) | $[1, 75]$ Mpc | standard for CBC PE |
| 1 | $\theta_{JN}$ (inclination) | Isotropic ($p \propto \sin\iota$) | $[0, \pi]$ | uniform in $\cos\iota$ |
| 2 | $\phi_c$ (phase) | Uniform | $[0, 2\pi]$ | |
| 3 | $\psi$ (polarisation) | Uniform | $[0, \pi]$ | |
| 4 | $\mathcal{M}_c$ (chirp mass) | Uniform | $[1.18, 1.20]\,M_\odot$ | |
| 5 | $q$ (mass ratio) | Uniform | $[0.5, 1.0]$ | standard LVK range |
| 6 | $t_c$ (coalescence time) | Uniform | $[-0.1, 0.1]\,\mathrm{s}$ | |
| 7 | $\chi_1$ (spin 1) | Uniform | $[-0.2, 0.2]$ | aligned BNS |
| 8 | $\chi_2$ (spin 2) | Uniform | $[-0.2, 0.2]$ | aligned BNS |
| 9 | $\Lambda_1$ (tidal 1) | Uniform | $[0, 2000]$ | |
| 10 | $\Lambda_2$ (tidal 2) | Uniform | $[0, 2000]$ | |

### Key fix in v4 (vs v3): phase aliasing bug

**v3 bug**: the data was pre-rotated by $e^{+2\pi i f(D-1)}$ before resampling to the coarse grid, but the template **still** applied $e^{-2\pi i f(t_d + \Delta t_c + (D-1))}$. This doubles the $(D-1)$ phase:
$$|d \cdot e^{+i\alpha} - h \cdot e^{-i(\beta+\alpha)}|^2 = |d - h \cdot e^{-i(\beta+2\alpha)}|^2 \neq |d - h \cdot e^{-i(\beta+\alpha)}|^2$$

**v4 fix**: the template now uses `timeshift = td + delta_tc` (without `duration - 1.0`), since that phase is already absorbed into the rotated data:
$$|d_{\rm rot} - h \cdot e^{-2\pi i f(t_d + \Delta t_c)}|^2 = |d - h \cdot e^{-2\pi i f(t_d + \Delta t_c + (D-1))}|^2 \quad \checkmark$$

## Environment setup (Colab / LIGO JupyterHub / fresh environment)

This cell installs all required packages and clones the repositories.

- **Google Colab**: Installs JAX (CUDA 12), clones `nuts_dynesty` and `mlgw_bns_jax`, and installs them.
- **LIGO JupyterHub**: Assumes `nuts_dynesty` is already installed in the active conda environment; only ensures `gwpy`, `corner`, and `h5py` are available.
- **Local**: Skip if everything is already installed.

In [1]:
import os, subprocess, sys, shutil

COLAB = "google.colab" in sys.modules
LIGO  = os.path.isdir("/cvmfs/oasis.opensciencegrid.org")   # LIGO JupyterHub marker

if COLAB:
    # ── Install JAX with CUDA 12 support ─────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Install other Python packages ────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py", "blackjax", "tqdm",
    ])

    # ── Clone & install nuts_dynesty ─────────────────────────────────
    NUTS_DIR = "/content/nuts_dynesty"
    if not os.path.isdir(NUTS_DIR):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/nuts_dynesty.git",
            NUTS_DIR,
        ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e", NUTS_DIR,
    ])

    # ── Clone mlgw_bns_jax (waveform model + loader) ────────────────
    MLGW_DIR = "/content/mlgw_bns_jax"
    if not os.path.isdir(MLGW_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "jax_mlgw_bns", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            MLGW_DIR,
        ])

    # Copy model + loader into working directory
    WORK_DIR = os.path.join(NUTS_DIR, "test_gw170817")
    os.chdir(WORK_DIR)

    for fname in ["jax_import_n_predict.py", "mlgw_bns_jax_model.h5"]:
        src = os.path.join(MLGW_DIR, fname)
        dst = os.path.join(WORK_DIR, fname)
        if os.path.isfile(src) and not os.path.isfile(dst):
            shutil.copy2(src, dst)

    print(f"Working directory: {os.getcwd()}")

elif LIGO:
    # On LIGO JupyterHub, nuts_dynesty should already be installed
    # in the active conda environment. Just ensure extras are present.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "corner", "gwpy", "h5py",
    ])
    os.chdir(os.path.dirname(os.path.abspath("__file__")))
    print(f"LIGO JupyterHub — Working directory: {os.getcwd()}")

else:
    print("Local environment — skipping setup.")

Local environment — skipping setup.


## Download GWOSC data (BayesWave-cleaned L1)

Downloads 1024 s of 4 kHz strain from GWOSC for H1, L1 and V1.

For **L1**, the scatter-light glitch near the merger is removed using the
official **BayesWave glitch subtraction** from
[DCC LIGO-T1700406-v3](https://dcc.ligo.org/LIGO-T1700406/public) —
the same cleaned data used for the GWTC-1 parameter estimation
(Abbott+ 2019, PRX 9, 011001).

The cleaned GWF covers GPS ≥ 1187008667 (553 s into our 1024 s window).
We splice: raw L1 for the earlier portion + BayesWave-cleaned for
the rest. The splice point is ~215 s before the trigger — far from
the glitch region.

**Skip if the cleaned files already exist.**

In [2]:
import os, sys, time, shutil
import numpy as np

_GPS_START = 1187008114
_DURATION  = 1024
_SRATE     = 4096
_DATA_DIR  = "gw170817_data"
os.makedirs(_DATA_DIR, exist_ok=True)

_DETECTORS = ["H1", "L1", "V1"]

# ── BayesWave-subtracted L1 data from DCC LIGO-T1700406-v3 ──────────
_DCC_GWF_URL = (
    "https://dcc.ligo.org/public/0144/T1700406/003/"
    "L-L1_CLEANED_HOFT_C02_T1700406_v3-1187008667-4096.gwf"
)
_DCC_CHANNEL = "L1:DCH-CLEAN_STRAIN_C02_T1700406_v3"
_DCC_GPS0    = 1187008667   # start of cleaned file
_DCC_SRATE   = 16384        # native sample rate of the GWF


def _ensure_gwf_backend():
    """Make sure at least one GWF reader is importable."""
    # Already available?
    for mod in ("frameCPP", "lalframe", "framel"):
        try:
            __import__(mod)
            return
        except ImportError:
            pass

    # Try conda-forge (works for Python 3.14+)
    conda = shutil.which("conda") or shutil.which("mamba")
    if conda:
        import subprocess
        for pkg in ("framel", "python-lalframe"):
            print(f"  Trying: {conda} install -c conda-forge {pkg}",
                  flush=True)
            ret = subprocess.call(
                [conda, "install", "-c", "conda-forge", "-y", "-q", pkg])
            if ret == 0:
                return

    # Try pip as last resort
    import subprocess
    for pkg in ("framel",):
        ret = subprocess.call(
            [sys.executable, "-m", "pip", "install", "-q", pkg])
        if ret == 0:
            return

    raise ImportError(
        "Cannot read GWF files.  Install a backend manually:\n"
        "  conda install -c conda-forge framel          # or\n"
        "  conda install -c conda-forge python-lalframe # or\n"
        "  conda install -c conda-forge lalsuite\n"
    )


def _read_gwf_channel(path, channel, start, end):
    """Read a single channel from a GWF file (try gwpy → framel)."""
    duration = end - start

    # ── gwpy (uses frameCPP or LALFrame under the hood) ──
    try:
        from gwpy.timeseries import TimeSeries
        ts = TimeSeries.read(path, channel, start=start, end=end)
        return np.asarray(ts.value, dtype=np.float64), float(ts.sample_rate.value)
    except Exception:
        pass

    # ── framel ──
    import framel
    vec = framel.frgetvect1d(path, channel, start, duration, 0)
    data = np.asarray(vec[0], dtype=np.float64)
    sr   = 1.0 / vec[3]          # vec[3] = sample spacing
    return data, sr


# ── Check if cleaned files already exist ─────────────────────────────
_all_exist = all(
    os.path.isfile(os.path.join(_DATA_DIR,
        f"{d[0]}-{d}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt"))
    for d in _DETECTORS
)

if _all_exist:
    print("Cleaned data files already exist — skipping download.")
else:
    from gwpy.timeseries import TimeSeries
    from scipy.signal import decimate as _decimate

    for det in _DETECTORS:
        out_file = os.path.join(_DATA_DIR,
            f"{det[0]}-{det}_BWCLEANED_4KHZ-{_GPS_START}-{_DURATION}.txt")

        if det == "L1":
            # ── L1: BayesWave-cleaned data from DCC ──────────────
            print("L1: building cleaned timeseries")
            t0 = time.time()

            # 1) Raw L1 from GWOSC (for the part before the cleaned file)
            print("  Downloading raw L1 from GWOSC...", flush=True)
            ts_raw = TimeSeries.fetch_open_data(
                "L1", _GPS_START, _GPS_START + _DURATION,
                sample_rate=_SRATE)

            # 2) Download BayesWave GWF from DCC (~1 GB, cached)
            gwf_local = os.path.join(_DATA_DIR,
                                     "L1_cleaned_bw_T1700406.gwf")
            if not os.path.isfile(gwf_local):
                import requests
                print("  Downloading BayesWave GWF from DCC (~1 GB)…",
                      flush=True)
                resp = requests.get(_DCC_GWF_URL, stream=True)
                resp.raise_for_status()
                with open(gwf_local, "wb") as fout:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fout.write(chunk)
                print(f"  Saved GWF "
                      f"({os.path.getsize(gwf_local)/1e6:.0f} MB)")
            else:
                print("  BayesWave GWF already cached.")

            # 3) Ensure a GWF backend is available
            _ensure_gwf_backend()

            # 4) Read the needed segment from the cleaned GWF
            _need_end = _GPS_START + _DURATION
            print("  Reading cleaned segment from GWF...", flush=True)
            bw_data, bw_sr = _read_gwf_channel(
                gwf_local, _DCC_CHANNEL, _DCC_GPS0, _need_end)

            # Resample 16384 → 4096 Hz if needed
            if int(round(bw_sr)) != _SRATE:
                factor = int(round(bw_sr)) // _SRATE
                print(f"  Resampling {int(bw_sr)} → {_SRATE} Hz "
                      f"(factor {factor})")
                bw_data = _decimate(bw_data, factor, ftype="iir",
                                    zero_phase=True)

            # 5) Splice: raw [GPS_START, DCC_GPS0] + cleaned [DCC_GPS0, end]
            n_raw = int((_DCC_GPS0 - _GPS_START) * _SRATE)
            strain = np.concatenate([
                ts_raw.value[:n_raw],   # raw (far from glitch)
                bw_data                 # BayesWave-cleaned
            ])
            n_expected = _DURATION * _SRATE
            assert len(strain) == n_expected, (
                f"L1 splice length mismatch: {len(strain)} "
                f"vs {n_expected}")

            with open(out_file, "w") as fw:
                fw.write("# BayesWave-cleaned L1 strain for GW170817\n")
                fw.write(f"# GPS [{_GPS_START}, {_GPS_START+_DURATION}], "
                         f"splice at GPS {_DCC_GPS0}\n")
                fw.write("# Before splice: raw GWOSC.  "
                         "After: DCC T1700406-v3 (BayesWave)\n")
                fw.write(f"# {_SRATE} samples per second\n")
                for val in strain:
                    fw.write(f"{val:.16e}\n")
            print(f"  → L1 done in {time.time()-t0:.1f}s")

        else:
            # ── H1 / V1: raw GWOSC (no glitch to fix) ───────────
            print(f"{det}: downloading {_DURATION}s from GWOSC…",
                  flush=True)
            t0 = time.time()
            ts = TimeSeries.fetch_open_data(
                det, _GPS_START, _GPS_START + _DURATION,
                sample_rate=_SRATE)
            with open(out_file, "w") as fw:
                fw.write(f"# {det}: raw GWOSC strain for GW170817\n")
                fw.write(f"# {_SRATE} samples per second\n")
                fw.write(f"# starting GPS {_GPS_START} "
                         f"duration {_DURATION}\n")
                for val in ts.value:
                    fw.write(f"{val:.16e}\n")
            print(f"  → saved in {time.time()-t0:.1f}s")

    print("All detectors ready.")

ModuleNotFoundError: No module named 'gwpy'

In [ ]:
from __future__ import annotations

import os, sys, time
from functools import partial

import numpy as np
from scipy.signal import welch as scipy_welch
from scipy.signal.windows import tukey
from scipy.interpolate import interp1d

# Use GPU if available on Colab, otherwise CPU
if "google.colab" not in sys.modules:
    os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

from nuts_dynesty import MCMCNestedSampler
from nuts_dynesty.gw_likelihood import (
    DETECTOR_DATABASE,
    DetectorData,
    antenna_pattern,
)
from nuts_dynesty.gw_utils import (
    mc_q_to_m1_m2,
    time_delay_from_earth_center,
)

print("JAX devices:", jax.devices())

## Load the mlgw_bns_jax waveform model

Load the JAX-based BNS waveform surrogate from its HDF5 file. This model takes `(q, λ₁, λ₂, χ₁, χ₂)` as input and returns `(h₊, h×)` polarizations.

In [ ]:
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

# Quick test: evaluate at a reference point
_test_params = jnp.array([1.0, 300.0, 300.0, 0.0, 0.0])
_test_freqs  = jnp.linspace(23.0, 2000.0, 100)
_hp_test, _hc_test = _mlgw_predict(
    _test_params, _test_freqs,
    total_mass=jnp.array(2.8),
    distance_mpc=jnp.array(40.0),
    inclination=jnp.array(0.3),
)
print(f"Model loaded — test hp shape: {_hp_test.shape}, max|hp|: {float(jnp.max(jnp.abs(_hp_test))):.3e}")

## Event parameters and analysis settings

In [ ]:
TRIGGER_TIME = 1187008882.43
SEGMENT_DURATION = 128.0        # paper-matching segment length
SAMPLING_RATE = 4096
F_LOWER = 23.0
F_UPPER = 2000.0
N_FREQ_POINTS = 3000            # low-resolution uniform grid
DATA_START_GPS = 1187008114     # 1024s data file start
DATA_DURATION = 1024            # total data length (s)

# Fixed sky location: NGC 4993 (EM counterpart of GW170817)
FIXED_RA  = 3.44616     # rad
FIXED_DEC = -0.408084   # rad

DATA_DIR = "gw170817_data"
OUTDIR = "results_test"
LABEL = "GW170817_nuts_lowgrid_v4"
os.makedirs(OUTDIR, exist_ok=True)

print(f"Segment duration: {SEGMENT_DURATION}s  →  original Δf = {1/SEGMENT_DURATION:.4f} Hz")
print(f"Low-res grid: {N_FREQ_POINTS} points in [{F_LOWER}, {F_UPPER}] Hz  →  Δf = {(F_UPPER - F_LOWER) / (N_FREQ_POINTS - 1):.4f} Hz")
print(f"Data: {DATA_DURATION}s starting GPS {DATA_START_GPS}")
print(f"Fixed sky: RA = {FIXED_RA:.5f} rad, Dec = {FIXED_DEC:.6f} rad  (NGC 4993)")

## Load cleaned data and build detector network

Build `DetectorData` (from `nuts_dynesty`) for each detector using the 1024 s of GWOSC strain.

With `SEGMENT_DURATION = 128 s`, we analyse a 128-s chunk ending 1 s after the trigger (SHARPy/LIGO convention: `seg_start = trigger - (duration - 1)`), and use the remaining ~896 s for Welch PSD estimation (~7 independent segments).

In [ ]:
# Cleaned 1024s files (L1 deglitched, H1/V1 as-is)
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")


def build_detector_data(ifo, data_file):
    """Build a DetectorData from GWOSC cleaned strain (128 s segment)."""
    strain = np.loadtxt(data_file, comments="#")
    dt = 1.0 / SAMPLING_RATE
    n_total = len(strain)

    lat, lon, gamma, zeta, elev = DETECTOR_DATABASE[ifo]

    # Segment selection: trigger at (duration - 1) from start
    seg_start_gps = TRIGGER_TIME - (SEGMENT_DURATION - 1.0)
    idx_start = int((seg_start_gps - DATA_START_GPS) * SAMPLING_RATE)
    n_seg = int(SEGMENT_DURATION * SAMPLING_RATE)

    assert idx_start >= 0 and idx_start + n_seg <= n_total, \
        f"Segment out of range for {ifo}"

    segment = strain[idx_start: idx_start + n_seg].copy()

    # Tukey window
    alpha_tukey = 0.4 / SEGMENT_DURATION
    window = tukey(n_seg, alpha_tukey)
    segment *= window
    window_norm = np.sqrt(n_seg / np.sum(window ** 2))

    # FFT
    freqs = np.fft.rfftfreq(n_seg, dt)
    sf = np.fft.rfft(segment) * window_norm * dt

    # PSD via Welch (off-source data)
    off_source = np.delete(strain, range(idx_start, idx_start + n_seg))
    f_psd, psd = scipy_welch(
        off_source, fs=SAMPLING_RATE, nperseg=n_seg,
        window=tukey(n_seg, alpha_tukey),
        noverlap=n_seg // 2,
    )
    psd_interp = interp1d(f_psd, psd, bounds_error=False, fill_value=np.inf)
    psd_vals = psd_interp(freqs)

    # Crop to analysis band
    df = 1.0 / SEGMENT_DURATION
    kmin = int(F_LOWER / df)
    kmax = min(int(F_UPPER / df) + 1, len(freqs))

    freqs_crop = jnp.array(freqs[kmin:kmax], dtype=jnp.float64)
    sf_crop = jnp.array(sf[kmin:kmax], dtype=jnp.complex128)
    psd_crop = jnp.array(psd_vals[kmin:kmax], dtype=jnp.float64)

    sigmasq = psd_crop * jnp.float64(dt) ** 2
    two_dt_over_n = 2.0 * dt / float(n_seg)

    return DetectorData(
        frequencies=freqs_crop,
        frequency_series=sf_crop,
        psd=psd_crop,
        sigmasq=sigmasq,
        two_delta_t_over_n=jnp.float64(two_dt_over_n),
        lat_deg=lat,
        lon_deg=lon,
        gamma_deg=gamma,
        zeta_deg=zeta,
        elevation=elev,
        trigger_time=TRIGGER_TIME,
        duration=SEGMENT_DURATION,
    )


print(f"\nBuilding detector network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
network = []
for ifo in ["H1", "L1", "V1"]:
    det = build_detector_data(ifo, data_files[ifo])
    network.append(det)
    print(f"  {ifo}: {len(det.frequencies)} freq bins, "
          f"PSD range [{float(det.psd.min()):.2e}, {float(det.psd.max()):.2e}]")
print(f"Network built in {time.time() - t0:.2f} s")

## Resample detector data to low-resolution frequency grid

The original FFT grid from the 128 s segment has df ~ 0.0078 Hz (~253k bins in [23, 2000] Hz). We now interpolate the data strain d(f) and the PSD Sn(f) onto a **uniform grid of 3000 points**, reducing the cost of each likelihood evaluation by ~84x.

- **Strain**: cubic interpolation of Re and Im parts separately
- **PSD**: cubic interpolation in log-space (ensures positivity)
- **Normalization**: the Whittle likelihood log L = -2 df sum |d_k - h_k|^2 / Sn(f_k) is preserved by setting `two_delta_t_over_n = 2 * df_new` and `sigmasq = psd_new`

Since `mlgw_bns_jax` can evaluate the waveform on **any** frequency grid, no interpolation of the template is needed.

### Phase rotation (critical for coarse grid)

Before interpolating, we rotate the data by $e^{+2\pi i f(D-1)}$ to absorb the large coalescence-time offset. This ensures the residual `d_rot - h` only involves the slowly-varying phase $e^{-2\pi i f(t_d + \Delta t_c)}$, which is safely within the Nyquist limit of the coarse grid ($\sim 0.76$ s). **The template must correspondingly use `timeshift = td + delta_tc` only** (see `project_waveform_bns`).

In [ ]:
from scipy.interpolate import interp1d as _interp1d

f_new = np.linspace(F_LOWER, F_UPPER, N_FREQ_POINTS)
df_new = f_new[1] - f_new[0]

print(f"Original FFT grid : {len(network[0].frequencies)} bins, "
      f"df = {1/SEGMENT_DURATION:.4f} Hz")
print(f"New uniform grid  : {N_FREQ_POINTS} bins, "
      f"df = {df_new:.4f} Hz  ({len(network[0].frequencies)/N_FREQ_POINTS:.0f}x reduction)\n")

# ── CRITICAL: phase rotation before resampling ───────────────────────
# The data sf contains a rapid phase factor exp(-2πi f (D-1)) from the
# FFT convention (coalescence at D-1 seconds from segment start).
# On the coarse grid (df~0.66 Hz), this phase wraps ~167x per bin,
# causing severe aliasing.  We absorb this phase into the data:
#
#   sf_rot(f) = sf(f) · exp(+2πi f (D-1))
#
# then the template only needs the small shift (td + Δtc), which is
# safely within the Nyquist limit of the coarse grid (~0.76 s).
# The likelihood |sf - h|² = |sf_rot - h_new|² is unchanged.
print(f"Phase rotation: absorbing exp(+2πi f × {SEGMENT_DURATION - 1:.0f}s) "
      f"into data before resampling")
print(f"  Nyquist time (coarse grid): {1/(2*df_new):.2f} s  "
      f"→  max |td+Δtc| ~ 0.13 s is safe\n")

_DET_NAMES = ["H1", "L1", "V1"]
network_lowgrid = []

for i, det in enumerate(network):
    f_orig = np.array(det.frequencies)
    
    # Rotate data to "trigger frame": remove (duration-1) phase
    phase_corr = 2.0 * np.pi * f_orig * (det.duration - 1.0)
    sf_rotated = np.array(det.frequency_series) * np.exp(1j * phase_corr)
    
    # Interpolate rotated frequency series (slowly varying now)
    sf_real = sf_rotated.real
    sf_imag = sf_rotated.imag
    
    sf_new_real = _interp1d(f_orig, sf_real, kind='cubic',
                            bounds_error=False, fill_value=0.0)(f_new)
    sf_new_imag = _interp1d(f_orig, sf_imag, kind='cubic',
                            bounds_error=False, fill_value=0.0)(f_new)
    sf_new = sf_new_real + 1j * sf_new_imag
    
    # Interpolate PSD in log-space to guarantee positivity
    psd_orig = np.array(det.psd)
    log_psd = np.log(np.where(psd_orig > 0, psd_orig, 1e-100))
    psd_new = np.exp(
        _interp1d(f_orig, log_psd, kind='cubic',
                  bounds_error=False, fill_value=np.log(1e-100))(f_new)
    )
    
    # Build new DetectorData with adjusted normalization:
    #   logL = -two_dt_over_n * sum |d-h|^2 / sigmasq
    #        = -2*df_new * sum |d-h|^2 / psd   (standard Whittle form)
    det_new = DetectorData(
        frequencies=jnp.array(f_new, dtype=jnp.float64),
        frequency_series=jnp.array(sf_new, dtype=jnp.complex128),
        psd=jnp.array(psd_new, dtype=jnp.float64),
        sigmasq=jnp.array(psd_new, dtype=jnp.float64),
        two_delta_t_over_n=jnp.float64(2.0 * df_new),
        lat_deg=det.lat_deg,
        lon_deg=det.lon_deg,
        gamma_deg=det.gamma_deg,
        zeta_deg=det.zeta_deg,
        elevation=det.elevation,
        trigger_time=det.trigger_time,
        duration=det.duration,
    )
    network_lowgrid.append(det_new)
    print(f"  {_DET_NAMES[i]}: {len(det_new.frequencies)} freq bins, "
          f"PSD range [{float(det_new.psd.min()):.2e}, {float(det_new.psd.max()):.2e}]")

# Replace the network with the low-resolution version
network = network_lowgrid
print(f"\nNetwork resampled to {N_FREQ_POINTS}-point uniform grid (trigger-frame).")

## Q-transform spectrograms

Verify the data quality: compare raw vs BayesWave-cleaned L1, and show all three cleaned detectors around the merger time.

In [ ]:
from gwpy.timeseries import TimeSeries
import matplotlib.pyplot as plt

MERGER_GPS = TRIGGER_TIME
WINDOW = 6.0
T_START_PLOT = MERGER_GPS - WINDOW / 2
T_END_PLOT   = MERGER_GPS + WINDOW / 2
F_MIN, F_MAX = 20.0, 800.0
Q_RANGE = (4, 64)

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN, F_MAX), qrange=Q_RANGE,
                            outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# ── L1 raw vs cleaned comparison ─────────────────────────────────────
raw_file = os.path.join(DATA_DIR,
    f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")

if os.path.isfile(raw_file):
    qt_raw = _qtransform(raw_file)
    qt_cln = _qtransform(data_files["L1"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw (with glitch)"),
                           (ax2, qt_cln, "L1 — BayesWave (DCC)")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                            qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7, label="Merger")
        ax.legend(loc="upper left"); ax.tick_params(labelsize=11)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1 glitch comparison (1024 s data)", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()
else:
    print(f"Raw L1 file not found ({raw_file}) — skipping glitch comparison.")

# ── All detectors cleaned ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                        qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} (cleaned)", fontsize=13)
    ax.tick_params(labelsize=11)
    ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (cleaned 1024 s data)",
             fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_all.png"), dpi=150)
plt.show()

print("Spectrograms saved.")

## Define BNS waveform projection, likelihood, and priors

We write a custom `project_waveform_bns` that:
1. Generates `(h₊, h×)` from `mlgw_bns_jax`
2. Applies antenna pattern `(F₊, F×)` at the fixed sky location
3. Applies the time-domain phase shift for detector time delay + coalescence offset

**v4 fix**: Since the data was pre-rotated by $e^{+2\pi i f(D-1)}$ in the resampling step, the template uses `timeshift = td + delta_tc` (without `duration - 1.0`). This avoids the phase-doubling bug in v3 and keeps timeshift within the Nyquist limit of the coarse grid.

The likelihood is the standard Whittle (frequency-domain) matched filter, identical to `nuts_dynesty.gw_likelihood` but using the BNS waveform.

**11 sampled parameters** (with literature-standard priors):
`[d_L, ι, φ_c, ψ, Mc, q, Δtc, χ₁, χ₂, Λ₁, Λ₂]`

**Non-uniform priors implemented via inverse CDF in `prior_transform`**:
- $d_L$: volumetric ($p \propto d_L^2$) → $d_L = (d_{\min}^3 + u (d_{\max}^3 - d_{\min}^3))^{1/3}$
- $\iota$: isotropic → $\iota = \arccos(1 - 2u)$

In [ ]:
# ── Waveform projection ──────────────────────────────────────────────────

def project_waveform_bns(
    params_11, det_freqs,
    det_lat_deg, det_lon_deg, det_gamma_deg, det_zeta_deg, det_elevation,
    trigger_time, duration,
):
    """Project mlgw_bns_jax waveform onto a single detector (fixed sky).

    params_11 = [d_L, iota, phi_c, psi, mc, q, delta_tc, chi1, chi2, lam1, lam2]

    NOTE (v4 fix): the data has been pre-rotated by exp(+2πi f (D-1)) in the
    resampling step. Therefore the template must NOT include (duration - 1.0)
    in the timeshift — only the small td + delta_tc component, which is
    safely within the Nyquist limit of the coarse frequency grid.
    """
    dist_mpc    = params_11[0]       # d_L directly in Mpc
    inclination = params_11[1]
    phi_c       = params_11[2]
    psi         = params_11[3]
    mc          = params_11[4]
    q           = params_11[5]
    delta_tc    = params_11[6]
    chi1        = params_11[7]
    chi2        = params_11[8]
    lambda_1    = params_11[9]
    lambda_2    = params_11[10]

    m1, m2 = mc_q_to_m1_m2(mc, q)
    total_mass = m1 + m2

    # mlgw_bns_jax polarizations
    # Convert q = m2/m1 (sampler) -> q_mlgw = m1/m2 >= 1 (mlgw_bns_jax convention)
    q_mlgw = 1.0 / q
    mlgw_params = jnp.array([q_mlgw, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, det_freqs,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )

    # Coalescence phase
    phase_factor = jnp.exp(-1j * phi_c)
    hp = hp * phase_factor
    hc = hc * phase_factor

    # Antenna pattern at fixed sky
    tc = trigger_time + delta_tc
    fplus, fcross = antenna_pattern(
        jnp.float64(FIXED_RA), jnp.float64(FIXED_DEC), psi, tc,
        det_lat_deg, det_lon_deg, det_gamma_deg, det_zeta_deg,
    )

    # Time delay from Earth center
    td = time_delay_from_earth_center(
        det_lat_deg, det_lon_deg, det_elevation,
        jnp.float64(FIXED_RA), jnp.float64(FIXED_DEC), tc,
    )

    # Phase shift: only td + delta_tc (the (D-1) phase is already in the data)
    timeshift = td + delta_tc
    phase_shift = 2.0 * jnp.pi * det_freqs * timeshift

    h = (fplus * hp + fcross * hc) * (
        jnp.cos(phase_shift) - 1j * jnp.sin(phase_shift)
    )
    return h


# ── Single-detector log-likelihood ──────────────────────────────────────

def log_likelihood_single_detector_bns(params_11, detector):
    """Whittle log-likelihood for a single detector (BNS waveform)."""
    h = project_waveform_bns(
        params_11, detector.frequencies,
        detector.lat_deg, detector.lon_deg,
        detector.gamma_deg, detector.zeta_deg, detector.elevation,
        detector.trigger_time, detector.duration,
    )
    residuals = detector.frequency_series - h
    sigmasq_safe = jnp.where(
        detector.sigmasq > 0, detector.sigmasq, jnp.ones_like(detector.sigmasq)
    )
    weighted = residuals / jnp.sqrt(sigmasq_safe)
    logL = -detector.two_delta_t_over_n * jnp.vdot(weighted, weighted).real
    return jnp.where(jnp.isfinite(logL), logL, -jnp.inf)


# ── Network log-likelihood ──────────────────────────────────────────────

def log_likelihood_network_bns(params_11, detectors):
    """Sum the log-likelihood over the H1+L1+V1 network."""
    total = jnp.zeros((), dtype=jnp.float64)
    for det in detectors:
        total = total + log_likelihood_single_detector_bns(params_11, det)
    return jnp.where(jnp.isfinite(total), total, -jnp.inf)


# ── Prior (literature-standard) ──────────────────────────────────────────
#
# Non-uniform priors implemented via inverse CDF in prior_transform:
#   - d_L:   volumetric  p(d_L) ∝ d_L²    →  d_L = (d_min³ + u·(d_max³ - d_min³))^(1/3)
#   - ι:     isotropic   p(ι) ∝ sin(ι)    →  ι   = arccos(1 - 2u)

# Distance bounds (Mpc)
D_L_MIN = 1.0
D_L_MAX = 75.0

# Pre-compute cubic constants for prior_transform
_d3_min = D_L_MIN ** 3
_d3_range = D_L_MAX ** 3 - _d3_min

# Uniform-prior parameters: [low, high] for each
# (used only for params with uniform prior)
_UNIFORM_LOW = jnp.array([
    0.0,          # [0]  placeholder (d_L handled separately)
    0.0,          # [1]  placeholder (iota handled separately)
    0.0,          # [2]  phi_c
    0.0,          # [3]  psi
    1.18,         # [4]  mc
    0.5,          # [5]  q  (standard LVK range)
    -0.1,         # [6]  delta_tc
    -0.2,         # [7]  chi1
    -0.2,         # [8]  chi2
    0.0,          # [9]  lambda_1
    0.0,          # [10] lambda_2
])

_UNIFORM_HIGH = jnp.array([
    0.0,          # [0]  placeholder
    0.0,          # [1]  placeholder
    2*jnp.pi,     # [2]  phi_c
    jnp.pi,       # [3]  psi
    1.20,         # [4]  mc
    1.0,          # [5]  q
    0.1,          # [6]  delta_tc
    0.2,          # [7]  chi1
    0.2,          # [8]  chi2
    2000.0,       # [9]  lambda_1
    2000.0,       # [10] lambda_2
])

PARAM_NAMES = [
    r"$d_L$", r"$\iota$", r"$\phi_c$", r"$\psi$",
    r"$\mathcal{M}_c$", r"$q$", r"$\Delta t_c$",
    r"$\chi_1$", r"$\chi_2$", r"$\Lambda_1$", r"$\Lambda_2$",
]
SHORT_NAMES = [
    "dL", "iota", "phic", "psi", "mc", "q", "dtc",
    "chi1", "chi2", "lambda1", "lambda2",
]


def prior_transform(u):
    """Map unit cube [0,1]^11 to physical parameter space.

    Implements standard LVK priors:
      - d_L:  volumetric  p(d_L) ∝ d_L²
      - ι:    isotropic   p(ι) ∝ sin(ι)    (uniform in cos ι)
      - all others: uniform
    """
    # Start with uniform mapping for all params
    theta = _UNIFORM_LOW + u * (_UNIFORM_HIGH - _UNIFORM_LOW)

    # [0] Distance: volumetric — p(d_L) ∝ d_L²
    d_L = jnp.power(_d3_min + u[0] * _d3_range, 1.0 / 3.0)

    # [1] Inclination: isotropic — uniform in cos(ι) over [0, π]
    iota = jnp.arccos(1.0 - 2.0 * u[1])

    # Assemble: overwrite indices 0 and 1
    theta = theta.at[0].set(d_L)
    theta = theta.at[1].set(iota)

    return theta


# ── Sanity check ────────────────────────────────────────────────────────

log_like = partial(log_likelihood_network_bns, detectors=network)

# Approximate GW170817 parameters
test_params = jnp.array([
    40.0,              # d_L ~ 40 Mpc
    jnp.pi / 6,       # inclination
    1.0,               # phi_c
    0.5,               # psi
    1.1976,            # mc
    0.9,               # q
    0.0,               # delta_tc
    0.0,               # chi1
    0.0,               # chi2
    300.0,             # lambda_1
    300.0,             # lambda_2
])

logL_check = log_like(test_params)
print(f"logL at approximate GW170817 params = {float(logL_check):.2f}")
print(f"  (should be ~O(-1000 to -100), NOT ~-600000)")
print(f"Fixed: RA = {FIXED_RA:.5f}, Dec = {FIXED_DEC:.6f}")
print(f"Distance: volumetric d_L ∈ [{D_L_MIN}, {D_L_MAX}] Mpc")
print(f"Inclination: isotropic (uniform in cos ι)")
print(f"Sampling {len(SHORT_NAMES)} parameters: {SHORT_NAMES}")

# Verify prior_transform roundtrip
u_mid = jnp.full(11, 0.5)
theta_mid = prior_transform(u_mid)
print(f"\nPrior transform at u=0.5:")
for name, val in zip(SHORT_NAMES, theta_mid):
    print(f"  {name:>10s} = {float(val):.6f}")

## Run the NUTS nested sampler

Using `MCMCNestedSampler` from `nuts_dynesty`. The first chunk triggers JIT compilation of the NUTS kernel (may take several minutes). Subsequent chunks reuse the compiled code.

Checkpoints are saved to disk after each chunk; set `dlogZ_stop` for early stopping when the remaining evidence contribution is negligible.

In [ ]:
NLIVE = 2000
MAX_ITER = 100000
NUM_MCMC_STEPS = 50
PROPOSAL_SCALE = 1.0
CHUNK_SIZE = 100
NUM_REPLACE = 10          # parallel dead-point replacement (GPU speedup)
DLOGZ_STOP = 0.01        # stop when remaining evidence < 1%
SEED = 42

print(f"Sampler: nlive={NLIVE}, max_iter={MAX_ITER}, "
      f"mcmc_steps={NUM_MCMC_STEPS}, num_replace={NUM_REPLACE}")
print(f"Chunk size: {CHUNK_SIZE}, dlogZ_stop={DLOGZ_STOP}")

sampler = MCMCNestedSampler(
    log_likelihood_fn=log_like,
    prior_transform=prior_transform,
    ndim=11,
    nlive=NLIVE,
    num_mcmc_steps=NUM_MCMC_STEPS,
    proposal_scale=PROPOSAL_SCALE,
    max_iter=MAX_ITER,
    num_replace=NUM_REPLACE,
)

print("Starting NUTS nested sampling — first chunk will JIT-compile...\n")
t0 = time.time()

result = sampler.run_chunked(
    jax.random.PRNGKey(SEED),
    chunk_size=CHUNK_SIZE,
    progress=True,
    output_dir=OUTDIR,
    dlogZ_stop=DLOGZ_STOP,
)

elapsed = time.time() - t0
logZ = float(result.logZ)
logZ_err = float(result.logZ_err)
print(f"\nDone in {elapsed:.1f} s")
print(f"log(Z) = {logZ:.2f} ± {logZ_err:.2f}")
print(f"Iterations completed: {int(result.num_iterations)}")

## Posterior samples and corner plot

Compute importance weights from the nested sampling dead points, resample, and produce a corner plot of all 11 parameters.

In [ ]:
from corner import corner

# ── Extract dead points and compute posterior weights ────────────────────
dead_logL = np.array(result.dead_logL)
dead_u = np.array(result.dead_points_u)

valid = np.isfinite(dead_logL) & (dead_logL > -1e29)
dead_logL_v = dead_logL[valid]
dead_u_v = dead_u[valid]
n_dead = len(dead_logL_v)

steps = np.arange(n_dead)
log_dX = np.log(1 - np.exp(-1.0 / NLIVE)) - steps / NLIVE
log_w = dead_logL_v + log_dX
log_w -= log_w.max()
weights = np.exp(log_w)
weights /= weights.sum()

n_eff = int(1.0 / np.sum(weights ** 2))
rng = np.random.default_rng(0)
indices = rng.choice(n_dead, size=max(n_eff, 50), p=weights)
samples_u = dead_u_v[indices]
samples = np.array(jax.vmap(prior_transform)(jnp.array(samples_u)))

print(f"Dead points: {n_dead}, effective posterior samples: {n_eff}")

# ── Save posterior ───────────────────────────────────────────────────────
samples_path = os.path.join(OUTDIR, "posterior_samples.npz")
np.savez(samples_path, samples=samples,
         param_names=SHORT_NAMES, logZ=logZ,
         logZ_err=logZ_err, n_eff=n_eff)
print(f"Saved to {samples_path}")

# ── Diagnostics ──────────────────────────────────────────────────────────
print(f"\n  {'Param':>10s}  {'Median':>10s}  {'16%':>10s}  {'84%':>10s}")
print("  " + "-" * 44)
for j, name in enumerate(SHORT_NAMES):
    p16 = np.percentile(samples[:, j], 16)
    p50 = np.percentile(samples[:, j], 50)
    p84 = np.percentile(samples[:, j], 84)
    print(f"  {name:>10s}  {p50:10.4f}  {p16:10.4f}  {p84:10.4f}")

# ── Corner plot ──────────────────────────────────────────────────────────
fig = corner(
    samples, show_titles=True,
    labels=PARAM_NAMES, title_kwargs={"fontsize": 10},
)
plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
fig.savefig(plot_path, dpi=150)
print(f"\nCorner plot saved to {plot_path}")
fig

## Paper-style corner plot (Figure 9)

Compute derived parameters from the posterior samples and produce a corner plot matching Figure 9 of [arXiv:2210.15684](https://arxiv.org/abs/2210.15684), showing only:
- $\mathcal{M}_c$ — chirp mass
- $q$ — mass ratio
- $\chi_\text{eff}$ — effective spin parameter
- $\tilde{\Lambda}$ — reduced tidal deformability
- $D_L$ — luminosity distance [Mpc]

Column indices for the 11-parameter samples:
`[0] dL, [1] incl, [2] phic, [3] pol, [4] mc, [5] q, [6] tc, [7] chi1, [8] chi2, [9] lambda_1, [10] lambda_2`

In [ ]:
# Extract raw sampled parameters (11-param layout)
dL_samples   = samples[:, 0]     # d_L directly in Mpc (volumetric prior)
mc_samples   = samples[:, 4]
q_samples    = samples[:, 5]
chi1_samples = samples[:, 7]
chi2_samples = samples[:, 8]
lam1_samples = samples[:, 9]
lam2_samples = samples[:, 10]

# Compute component masses
m1_samples = np.zeros(len(mc_samples))
m2_samples = np.zeros(len(mc_samples))
for i in range(len(mc_samples)):
    m1_samples[i], m2_samples[i] = mc_q_to_m1_m2(
        jnp.float64(mc_samples[i]), jnp.float64(q_samples[i])
    )

# chi_eff = (m1*chi1 + m2*chi2) / (m1 + m2)
chi_eff_samples = (m1_samples * chi1_samples + m2_samples * chi2_samples) / (m1_samples + m2_samples)

# Lambda_tilde (reduced tidal deformability)
M_samples = m1_samples + m2_samples
lambda_tilde_samples = (16.0 / 13.0) * (
    (m1_samples + 12.0 * m2_samples) * m1_samples**4 * lam1_samples
    + (m2_samples + 12.0 * m1_samples) * m2_samples**4 * lam2_samples
) / M_samples**5

# Build the 5-parameter array for the corner plot
paper_samples = np.column_stack([
    mc_samples,
    q_samples,
    chi_eff_samples,
    lambda_tilde_samples,
    dL_samples,
])

paper_labels = [
    r"$\mathcal{M}_c$ $[M_\odot]$",
    r"$q$",
    r"$\chi_{\rm eff}$",
    r"$\tilde{\Lambda}$",
    r"$D_L$ [Mpc]",
]

fig_paper = corner(
    paper_samples, show_titles=True,
    labels=paper_labels,
    title_kwargs={"fontsize": 12},
    quantiles=[0.05, 0.5, 0.95],
    levels=(0.5, 0.9),
    fill_contours=True,
    color="tab:orange",
)
plot_path_paper = os.path.join(OUTDIR, f"{LABEL}_corner_paper.png")
fig_paper.savefig(plot_path_paper, dpi=150)
print(f"Saved to {plot_path_paper}")
fig_paper